# 04 — Padrões Estruturais de Falha

**Objetivo:** catalogar sistematicamente os modos de falha em todos os 96 sub-fluxos.  
Este notebook transforma os dados em um **mapa de falhas** que alimenta o ICE scoring do notebook 05.

**Cinco padrões detectados:**

| Código | Padrão | Definição operacional |
|---|---|---|
| P1 | Try-again loop | Ação `try-again` presente ≥1x na conversa |
| P2 | Verification loop | `verify-identity` aparece >1x (agente pediu verificação repetida) |
| P3 | Escalation inesperada | `notify-team` como última ação E subflow ≠ `out_of_stock_general` |
| P4 | Repetição de ação | Qualquer ação take_action aparece ≥3x |
| P5 | Outlier de comprimento | n_turns > média_subflow + 1,5 × desvio |

Uma conversa pode ter mais de um padrão.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

pd.set_option('display.float_format', '{:.1f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROC_DIR    = Path('../data/processed')
FIGURES_DIR = Path('../reports/figures')

df  = pd.read_parquet(PROC_DIR / 'conversations_patterns.parquet')
sf  = pd.read_csv(PROC_DIR / 'subflow_patterns_ice.csv')
print(f"Conversas: {len(df):,} | Subflows: {sf['subflow'].nunique()}")

## 1. Prevalência Global dos Padrões

In [ ]:
pattern_cols = {
    'p_try_again':   'P1 — Try-again loop',
    'p_verify_loop': 'P2 — Verification loop',
    'p_escalation':  'P3 — Escalation inesperada',
    'p_action_rep':  'P4 — Repetição de ação',
    'p_long_outlier':'P5 — Outlier de comprimento',
    'any_pattern':   'Qualquer padrão',
}

prev = {label: df[col].mean() * 100 for col, label in pattern_cols.items()}

fig, ax = plt.subplots(figsize=(9, 4))
labels = list(prev.keys())
values = list(prev.values())
colors = ['#d62728','#ff7f0e','#9467bd','#8c564b','#e377c2','#17becf']
bars = ax.barh(labels, values, color=colors, edgecolor='none')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%  ({int(val/100*len(df)):,})', va='center', fontsize=9)
ax.set_xlabel('% das conversas')
ax.set_title('Prevalência de Padrões Estruturais de Falha', fontweight='bold')
ax.set_xlim(0, 28)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_pattern_prevalence.png', bbox_inches='tight')
plt.show()

for label, pct in prev.items():
    print(f"  {label:35s}: {pct:.1f}%  ({int(pct/100*len(df)):,})")

## 2. Mapa de Calor: Padrão × Flow

In [ ]:
flow_patterns = df.groupby('flow')[list(pattern_cols.keys())[:-1]].mean() * 100
flow_patterns.columns = ['P1','P2','P3','P4','P5']
flow_patterns = flow_patterns.sort_values('P1', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(flow_patterns, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% conversas com padrão'})
ax.set_title('Padrões de Falha (%) por Flow', fontweight='bold', fontsize=12)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_pattern_heatmap_flow.png', bbox_inches='tight')
plt.show()

## 3. Drill-down: troubleshoot_site — O Epicentro

In [ ]:
ts = sf[sf['flow'] == 'troubleshoot_site'][[
    'subflow','n','any_pattern','p_try_again','p_verify_loop',
    'p_escalation','p_long_outlier','turns_mean'
]].sort_values('any_pattern', ascending=False)
print(ts.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Stacked bar: composição de padrões
subs = ts['subflow'].values
p1 = ts['p_try_again'].values
p2 = ts['p_verify_loop'].values
p5 = ts['p_long_outlier'].values

axes[0].bar(subs, p1, label='P1 try-again', color='#d62728')
axes[0].bar(subs, p2, bottom=p1, label='P2 verify loop', color='#ff7f0e')
axes[0].bar(subs, p5, bottom=p1+p2, label='P5 outlier', color='#e377c2')
axes[0].set_ylabel('% conversas')
axes[0].set_title('Composição dos Padrões', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].set_xticklabels(subs, rotation=15, ha='right')

# Turnos médios
colors_ts = ['#d62728' if v > 50 else '#aec7e8' for v in ts['any_pattern']]
axes[1].bar(subs, ts['turns_mean'].values, color=colors_ts, edgecolor='none')
axes[1].axhline(df['n_turns'].mean(), color='navy', ls='--', lw=1.5,
                label=f'Média global ({df["n_turns"].mean():.0f} turnos)')
axes[1].set_ylabel('Turnos médios')
axes[1].set_title('Comprimento Médio', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_xticklabels(subs, rotation=15, ha='right')

plt.suptitle('troubleshoot_site — Análise por Subflow', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_troubleshoot_site_patterns.png', bbox_inches='tight')
plt.show()

## 4. Padrão P1 em Detalhe: O Loop de Trial-and-Error

In [ ]:
p1_df = df[df['p_try_again'] == 1]
print(f"Conversas com P1 (try-again loop): {len(p1_df):,}")
print(f"\nDistribuição por flow:")
flow_p1 = (p1_df.groupby('flow')['convo_id'].count() / len(p1_df) * 100).sort_values(ascending=False)
for flow, pct in flow_p1.items():
    n = int(pct / 100 * len(p1_df))
    print(f"  {flow:25s}: {pct:.1f}%  ({n} conversas)")

print(f"\nNúmero de ciclos try-again por conversa:")
print(df[df['p_try_again']==1]['n_try_again'].value_counts().sort_index())

## 5. Padrão P2 em Detalhe: Verification Loop

In [ ]:
p2_df = df[df['p_verify_loop'] == 1]
print(f"Conversas com P2 (verification loop): {len(p2_df):,}")
print(f"\nDistribuição por flow:")
flow_p2 = (p2_df.groupby('flow')['convo_id'].count() / len(p2_df) * 100).sort_values(ascending=False)
for flow, pct in flow_p2.items():
    n = int(pct / 100 * len(p2_df))
    print(f"  {flow:25s}: {pct:.1f}%  ({n} conversas)")

print(f"\nSubflows mais afetados por P2:")
sf_p2 = sf[sf['p_verify_loop'] > 5][['flow','subflow','n','p_verify_loop']].sort_values('p_verify_loop', ascending=False)
print(sf_p2.to_string(index=False))

## 6. Sumário do Notebook 04

### Taxonomia de Falhas

| Padrão | Prevalência | Principal flow afetado | Mecanismo |
|---|---|---|---|
| P1 — Try-again loop | **7,0%** (704 conv.) | troubleshoot_site (96% dos casos) | Agente sugere solução genérica → falha → retry |
| P2 — Verification loop | 2,3% (227 conv.) | manage_account (50% dos casos) | Identidade pedida múltiplas vezes no mesmo atendimento |
| P3 — Escalation inesperada | 2,0% (198 conv.) | subscription_inquiry | Bot incapaz de resolver; escala para equipe |
| P4 — Repetição de ação | 1,6% (156 conv.) | manage_account | Mesma ação sistêmica executada 3+ vezes |
| P5 — Outlier de comprimento | **7,9%** (792 conv.) | Distribuído | Conversa 50%+ mais longa que a média do subflow |
| **Qualquer padrão** | **17,7%** (1.779 conv.) | — | — |

### Concentração do Problema

- **96% do P1 (try-again)** está concentrado em 1 flow: `troubleshoot_site`
- Dentro do flow, 4 subflows capturam 100% do sinal: `shopping_cart`, `credit_card`, `slow_speed`, `search_results`
- `shopping_cart` e `credit_card` juntos = **60% de todos os try-again loops do dataset**

**Próximo passo:** notebook 05 — ICE scoring para priorizar qual subflow vira o experimento A/B.